## Setup & Data Loading
Load transaction and auction CSVs, parse datetimes, and filter to the analysis period.

In [ ]:
import pandas as pd
import glob
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats

# Read all CSV files from the specified directory
csv_files = glob.glob(f"data/case_study/*_pnl.csv")
txs = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
auctions = pd.read_csv("data/case_study/timeboost_auctions_casestudy.csv")

# Drop any auction-derived columns that may have been added in a previous run
auction_cols = ['merged', 'auction_round', 'winner_address', 'winner_name',
                'top_bid_eth', 'paid_bid_eth', 'express_lane_controller_address',
                'round_start_time', 'round_end_time', 'top_bid_usd', 'paid_bid_usd']
txs = txs.drop(columns=[c for c in auction_cols if c in txs.columns])

txs["timeboosted"] = txs["timeboosted"].fillna(False)

txs["block_time"] = pd.to_datetime(txs["block_time"], utc=True).dt.tz_convert(None)
auctions["round_start_time"] = pd.to_datetime(auctions["round_start_time"], utc=True)
auctions["round_end_time"]   = pd.to_datetime(auctions["round_end_time"], utc=True)

txs = txs.sort_values("block_time")
auctions = auctions.sort_values("round_start_time")

In [ ]:
ZONE_LABELS = [
    (None,                                           pd.Timestamp("2026-02-12 20:30:51", tz="UTC"), "Pre-Kairos"),
    (pd.Timestamp("2026-02-12 20:30:51", tz="UTC"), pd.Timestamp("2026-02-18 20:01:52", tz="UTC"), "Kairos"),
    (pd.Timestamp("2026-02-18 20:01:52", tz="UTC"), pd.Timestamp("2026-02-24 01:16:50", tz="UTC"), "Reserve Price\nAdaptation"),
    (pd.Timestamp("2026-02-24 01:16:50", tz="UTC"), None,                                          "Steady State"),
]

ZONE_COLORS = ["#d0e8f5", "#fde8cc", "#d8f0d8", "#f5d8e8"]

# Reserve price per zone (ETH)
RESERVE_PRICE_ETH = {
    "Pre-Kairos":               0.001,
    "Kairos":                   0.001,
    "Reserve Price\nAdaptation":0.0075,
    "Steady State":             0.001,
}

def _assign_zone(d):
    """Assign a Python date to a zone using UTC-precise ZONE_LABELS boundaries."""
    ts = pd.Timestamp(d, tz="UTC")
    for z_start, z_end, label in ZONE_LABELS:
        if (z_start is None or ts >= z_start) and (z_end is None or ts < z_end):
            return label
    return "Unknown"

def _add_reserve_price_line(ax, t_min, t_max):
    """Overlay reserve price as a step line on a tz-naive datetime x-axis (ETH units)."""
    labeled = False
    for z_start, z_end, z_label in ZONE_LABELS:
        x0 = t_min if z_start is None else z_start.tz_convert(None).to_pydatetime()
        x1 = t_max if z_end   is None else z_end.tz_convert(None).to_pydatetime()
        rp = RESERVE_PRICE_ETH[z_label]
        ax.hlines(rp, x0, x1,
                  colors="crimson", linewidth=1.8, linestyle="--", zorder=5,
                  label="Reserve Price" if not labeled else "_nolegend_")
        labeled = True


## Price Feeds & Volatility
Load Binance 1-second price data and compute rolling ETH volatility.

In [ ]:
binance_btcusd = pd.read_csv(f"data/case_study/prices/BTCUSDT-1s-merged.csv")
binance_ethusd = pd.read_csv(f"data/case_study/prices/ETHUSDT-1s-merged.csv")
binance_arbusd = pd.read_csv(f"data/case_study/prices/ARBUSDT-1s-merged.csv")

cols = [
    "open_time_us", "open", "high", "low", "close", "volume",
    "close_time_us", "quote_vol", "trades", "taker_base", "taker_quote", "ignore"
]

binance_arbusd.columns = cols
binance_ethusd.columns = cols
binance_btcusd.columns = cols

HORIZONS = [0, 5, 10, 300, 600, 1800]
STABLECOINS = {"USDC", "USDâ‚®0"}

def prepare_pricefeed(df, token):
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["open_time_us"] / 1e6, unit="s", utc=True)
    df["midprice"] = (df["high"] + df["low"]) / 2
    df = df[["timestamp", "midprice"]].sort_values("timestamp").reset_index(drop=True)
    df = df.rename(columns={"midprice": f"{token}_mid"})
    return df

binance_arbusd = prepare_pricefeed(binance_arbusd, "ARB")
binance_ethusd = prepare_pricefeed(binance_ethusd, "ETH")
binance_btcusd = prepare_pricefeed(binance_btcusd, "BTC")

pricefeeds = {
    "ARB": binance_arbusd.sort_values("timestamp").reset_index(drop=True),
    "ETH": binance_ethusd.sort_values("timestamp").reset_index(drop=True),
    "BTC": binance_btcusd.sort_values("timestamp").reset_index(drop=True),
}

In [ ]:
eth = binance_ethusd.sort_values("timestamp").copy()
eth["timestamp"] = pd.to_datetime(eth["timestamp"], utc=True)
eth["log_ret"] = np.log(eth["ETH_mid"] / eth["ETH_mid"].shift(1))

# 30-min rolling window (1800 seconds)
eth["vol_30m"] = eth["log_ret"].rolling(1800).std()

daily_vol = eth.groupby(eth["timestamp"].dt.date)["log_ret"].std()

daily_vol = daily_vol.reset_index().rename(columns={"log_ret": "daily_vol"})

daily_vol["date"] = pd.to_datetime(daily_vol["timestamp"], utc=True).dt.date

# # Daily volatility = mean of intraday rolling vol
# daily_vol = (
#     eth.assign(date=eth["timestamp"].dt.date)
#        .groupby("date")["vol_30m"]
#        .mean()
#        .reset_index()
#        .rename(columns={"vol_30m": "daily_vol"})
# )

## Data Enrichment
Merge ETH price into auctions, compute USD bid amounts, and join auction info to transactions.

In [ ]:
# Calculate USD for top_bid_eth and paid_bid_eth, using the eth price at auction start time
auctions = auctions.merge(
    binance_ethusd.rename(columns={"ETH_mid": "eth_price_at_auction_start"}),
    left_on="round_start_time",
    right_on="timestamp",
    how="left"
)
auctions["top_bid_usd"]  = auctions["top_bid_eth"]  * auctions["eth_price_at_auction_start"]
auctions["paid_bid_usd"] = auctions["paid_bid_eth"] * auctions["eth_price_at_auction_start"]

# Drop existing auction columns from txs if present
for col in ["round_start_time", "round_end_time", "auction_round", "top_bid_usd", "paid_bid_usd"]:
    if col in txs.columns:
        txs = txs.drop(columns=[col])

# Update txs with auction info using range join on block_time within [round_start_time, round_end_time]
# auctions timestamps are tz-aware (UTC); block_time is tz-naive â€” align both to tz-naive UTC
auctions_tz_naive = auctions.copy()
auctions_tz_naive["round_start_time"] = auctions_tz_naive["round_start_time"].dt.tz_convert(None)
auctions_tz_naive["round_end_time"]   = auctions_tz_naive["round_end_time"].dt.tz_convert(None)

txs_enriched = pd.merge_asof(
    txs.sort_values("block_time"),
    auctions_tz_naive[["round_start_time", "round_end_time", "auction_round", "top_bid_usd", "paid_bid_usd", "winner_name"]].sort_values("round_start_time"),
    left_on="block_time",
    right_on="round_start_time",
    direction="backward"
)
# Nullify rows where block_time falls outside the matched round's window
mask = (
    txs_enriched["block_time"] >= txs_enriched["round_start_time"]
) & (
    txs_enriched["block_time"] <= txs_enriched["round_end_time"]
)
txs_enriched.loc[~mask, ["auction_round", "top_bid_usd", "paid_bid_usd", "round_start_time", "round_end_time"]] = None
txs = txs_enriched

txs["time_since_round_start"] = (txs["block_time"] - txs["round_start_time"]).dt.total_seconds()
txs["time_since_minute_start"] = (txs["block_time"] - txs["block_time"].dt.floor("T")).dt.total_seconds()

In [ ]:
# ETH price at block_time via merge_asof (last price at or before block_time)
# binance_ethusd timestamps are tz-aware UTC; block_time is tz-naive UTC â€” strip tz for alignment
eth_prices = binance_ethusd[["timestamp", "ETH_mid"]].copy()
eth_prices["timestamp"] = eth_prices["timestamp"].dt.tz_convert(None)

if "tx_fee_usd" in txs.columns:
    txs = txs.drop(columns=["tx_fee_usd", "eth_price_at_tx"], errors="ignore")

txs = pd.merge_asof(
    txs.sort_values("block_time"),
    eth_prices.rename(columns={"ETH_mid": "eth_price_at_tx"}).sort_values("timestamp"),
    left_on="block_time",
    right_on="timestamp",
    direction="backward",
)
txs["tx_fee_usd"] = txs["tx_fee_eth"] * txs["eth_price_at_tx"]


In [ ]:
SEARCHERS = {
    "Wintermute": [
        '0xcb43d843f6cadf4f4844f3f57032468aadd9b95c',
        '0x27920e8039d2b6e93e36f5d5f53b998e2e631a70'
    ],
    "Selini": [
        '0xee2e7bbb67676292af2e31dffd1fea2276d6c7ba'
    ]
}
ALL_SEARCHER_ADDRS = [a for addrs in SEARCHERS.values() for a in addrs]

wm_txs = txs[txs["tx_to_address"].isin(SEARCHERS["Wintermute"])].copy()
sel_txs = txs[txs["tx_to_address"].isin(SEARCHERS["Selini"])].copy()

wm_txs["date"] = wm_txs["block_time"].dt.date
sel_txs["date"] = sel_txs["block_time"].dt.date

## Auction Winners

In [ ]:
print(auctions["winner_name"].value_counts())

In [ ]:
WINNER_COLORS = {
    "Wintermute": "#4C72B0",
    "Selini":     "#DD8452",
    "Kairos":     "#55A868",
}

# Daily winner counts, reindexed to full date range so gaps show as zero
auctions["date"] = auctions["round_start_time"].dt.tz_convert(None).dt.date
daily_winners = (
    auctions.groupby(["date", "winner_name"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=list(WINNER_COLORS.keys()), fill_value=0)
)

full_index = pd.date_range(
    start=daily_winners.index.min(),
    end=daily_winners.index.max(),
    freq="D",
).date
daily_winners = daily_winners.reindex(full_index, fill_value=0)

fig, ax = plt.subplots(figsize=(14, 5))

daily_winners.plot(
    kind="bar",
    ax=ax,
    color=[WINNER_COLORS[c] for c in daily_winners.columns],
    width=0.8,
    alpha=0.9,
    zorder=3,
)

# Zone shading
dates = list(daily_winners.index)
n = len(dates)

def date_to_x(ts):
    d = ts.tz_convert(None).date()
    for i, bar_date in enumerate(dates):
        if bar_date >= d:
            return i - 0.5
    return n - 0.5

for i, (z_start, z_end, z_label) in enumerate(ZONE_LABELS):
    x0 = -0.5      if z_start is None else date_to_x(z_start)
    x1 = n - 0.5   if z_end   is None else date_to_x(z_end)
    ax.axvspan(x0, x1, alpha=0.18, color=ZONE_COLORS[i], zorder=0)
    ymax = ax.get_ylim()[1]
    ax.text((x0 + x1) / 2, ymax * 0.97,
            z_label, ha="center", va="top", fontsize=10,
            color="0.3", style="italic")

ax.set_title("Daily Auction Round Winners Over Time", fontsize=14)
ax.set_xlabel("")
ax.set_ylabel("Number of rounds won")
ax.tick_params(axis="x", rotation=90)
ax.legend(title="Winner")
ax.grid(axis="y", linestyle="--", alpha=0.4)
fig.tight_layout()
plt.savefig("figures/auction_winners_over_time.pdf", bbox_inches="tight", dpi=300)
plt.show()


In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 5))

auctions_sorted = auctions.sort_values("round_start_time").copy()
ts = auctions_sorted["round_start_time"].dt.tz_convert(None)

ax1.plot(ts, auctions_sorted["top_bid_eth"],
         label="Top Bid", color="#4C72B0", linewidth=0.5, alpha=0.8)
ax1.plot(ts, auctions_sorted["paid_bid_eth"],
         label="Paid Bid", color="#DD8452", linewidth=0.5, alpha=0.8)
ax1.set_ylabel("Bid (ETH)")
ax1.grid(axis="y", linestyle="--", alpha=0.4)

t_min, t_max = ts.min(), ts.max()

_add_reserve_price_line(ax1, t_min, t_max)

# Volatility on right axis
ax2 = ax1.twinx()
vol_ts = pd.to_datetime(daily_vol["date"])
ax2.plot(vol_ts, daily_vol["daily_vol"],
         color="#2ca02c", linewidth=2, alpha=0.85, label="ETH Daily Vol")
ax2.set_ylabel("ETH Daily Volatility", color="#2ca02c", fontsize=10)
ax2.tick_params(axis="y", labelcolor="#2ca02c")
ax2.yaxis.set_label_position("right")

# Zone shading
ymax = ax1.get_ylim()[1]
for i, (z_start, z_end, z_label) in enumerate(ZONE_LABELS):
    x0 = t_min if z_start is None else z_start.tz_convert(None).to_pydatetime()
    x1 = t_max if z_end   is None else z_end.tz_convert(None).to_pydatetime()
    ax1.axvspan(x0, x1, alpha=0.18, color=ZONE_COLORS[i], zorder=0)
    mid = x0 + (x1 - x0) / 2
    ax1.text(mid, ymax * 0.97, z_label, ha="center", va="top",
             fontsize=9, color="0.3", style="italic")

lines1, labs1 = ax1.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labs1 + labs2, fontsize=9, loc="upper left")

ax1.set_xlabel("Date")
ax1.set_title("Top Bid vs Paid Bid per Auction Round")
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig("figures/bids_per_round.pdf", bbox_inches="tight", dpi=300)
plt.show()


In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 5))

auctions_sorted = auctions.sort_values("round_start_time").copy()
auctions_sorted["bid_gap_eth"] = auctions_sorted["top_bid_eth"] - auctions_sorted["paid_bid_eth"]
ts = auctions_sorted["round_start_time"].dt.tz_convert(None)

ax1.plot(ts, auctions_sorted["bid_gap_eth"],
         color="#4C72B0", linewidth=0.5, alpha=0.8, label="Bid Gap (ETH)")
ax1.set_ylabel("Bid Gap (ETH)")
ax1.grid(axis="y", linestyle="--", alpha=0.4)

t_min, t_max = ts.min(), ts.max()

_add_reserve_price_line(ax1, t_min, t_max)

# Volatility on right axis
ax2 = ax1.twinx()
vol_ts = pd.to_datetime(daily_vol["date"])
ax2.plot(vol_ts, daily_vol["daily_vol"],
         color="#2ca02c", linewidth=2, alpha=0.85, label="ETH Daily Vol")
ax2.set_ylabel("ETH Daily Volatility", color="#2ca02c", fontsize=10)
ax2.tick_params(axis="y", labelcolor="#2ca02c")
ax2.yaxis.set_label_position("right")

# Zone shading
ymax = ax1.get_ylim()[1]
for i, (z_start, z_end, z_label) in enumerate(ZONE_LABELS):
    x0 = t_min if z_start is None else z_start.tz_convert(None).to_pydatetime()
    x1 = t_max if z_end   is None else z_end.tz_convert(None).to_pydatetime()
    ax1.axvspan(x0, x1, alpha=0.18, color=ZONE_COLORS[i], zorder=0)
    mid = x0 + (x1 - x0) / 2
    ax1.text(mid, ymax * 0.97, z_label, ha="center", va="top",
             fontsize=9, color="0.3", style="italic")

lines1, labs1 = ax1.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labs1 + labs2, fontsize=9, loc="upper left")

ax1.set_xlabel("Date")
ax1.set_title("Bid Gap per Auction Round vs ETH Volatility")
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig("figures/bid_gap_per_round.pdf", bbox_inches="tight", dpi=300)
plt.show()


In [ ]:
wm_addrs  = set(SEARCHERS["Wintermute"])
sel_addrs = set(SEARCHERS["Selini"])

tb_txs = txs[txs["timeboosted"] == True].copy()
tb_txs["is_wm"]    = tb_txs["tx_to_address"].isin(wm_addrs)
tb_txs["is_sel"]   = tb_txs["tx_to_address"].isin(sel_addrs)
tb_txs["is_other"] = ~tb_txs["tx_to_address"].isin(wm_addrs) & ~tb_txs["tx_to_address"].isin(sel_addrs)

round_presence = (
    tb_txs.groupby("auction_round")
    .agg(has_wm=("is_wm", "any"), has_sel=("is_sel", "any"), has_other=("is_other", "any"))
    .reset_index()
)

aug = auctions.merge(round_presence, on="auction_round", how="left")
for col in ["has_wm", "has_sel", "has_other"]:
    aug[col] = aug[col].fillna(False)
aug["zone"] = aug["round_start_time"].dt.tz_convert(None).dt.date.apply(_assign_zone)

# Mutually exclusive categories (exhaustive: sums to 100%)
aug["cat_only_wm"]  =  aug["has_wm"] & ~aug["has_sel"] & ~aug["has_other"]
aug["cat_only_sel"] = ~aug["has_wm"] &  aug["has_sel"] & ~aug["has_other"]
aug["cat_both"]     =  aug["has_wm"] &  aug["has_sel"] & ~aug["has_other"]
aug["cat_mixed"]    =  aug["has_other"]   # any TB tx outside WM+Sel (Â±WM/Sel also present)
aug["cat_neither"]  = ~aug["has_wm"] & ~aug["has_sel"] & ~aug["has_other"]

tb_cats = [
    ("Only WM",    "cat_only_wm"),
    ("Only Sel",   "cat_only_sel"),
    ("Both",       "cat_both"),
    ("Mixed",      "cat_mixed"),
    ("Neither",    "cat_neither"),
]

zone_order = [label for _, _, label in ZONE_LABELS]
col_w = 24

for winner in ["Wintermute", "Selini", "Kairos"]:
    w_mask = aug["winner_name"] == winner
    print(f"=== {winner} wins ===")
    print(f"{'TB presence':<14}", end="")
    for z in zone_order:
        print(f"  {z.replace(chr(10), ' '):>{col_w}}", end="")
    print()
    print("-" * (14 + (col_w + 2) * len(zone_order)))
    for cat_label, cat_col in tb_cats:
        print(f"{cat_label:<14}", end="")
        for z in zone_order:
            z_mask   = aug["zone"] == z
            n_entity = (w_mask & z_mask).sum()
            n_case   = (w_mask & z_mask & aug[cat_col]).sum()
            pct      = 100 * n_case / n_entity if n_entity > 0 else float("nan")
            print(f"  {f'{pct:.1f}% ({n_case}/{n_entity})':>{col_w}}", end="")
        print()
    print()


In [ ]:
# Daily total paid bid = Arbitrum revenue from Timeboost
daily_paid = (
    auctions.groupby("date")["paid_bid_usd"]
    .sum()
    .reset_index()
    .rename(columns={"paid_bid_usd": "daily_paid_usd"})
)

df_pv = daily_paid.merge(daily_vol, on="date", how="inner").dropna()



df_pv["zone"] = df_pv["date"].apply(_assign_zone)

fig, ax = plt.subplots(figsize=(8, 5))

print("Arbitrum Revenue (Daily Total Paid Bid) vs ETH Daily Volatility â€” by Zone")
print(f"{'Zone':<32}  {'r':>7}  {'p':>8}  n")
print("-" * 57)

for (z_start, z_end, z_label), z_color in zip(ZONE_LABELS, ZONE_COLORS):
    sub = df_pv[df_pv["zone"] == z_label]
    if len(sub) < 2:
        continue
    label_flat = z_label.replace("\n", " ")
    ax.scatter(sub["daily_vol"], sub["daily_paid_usd"],
               label=label_flat, color=z_color, edgecolors="0.35",
               linewidths=0.5, s=60, alpha=0.9, zorder=3)
    m, b = np.polyfit(sub["daily_vol"], sub["daily_paid_usd"], 1)
    xs = np.linspace(sub["daily_vol"].min(), sub["daily_vol"].max(), 100)
    ax.plot(xs, m * xs + b, color=z_color, linewidth=1.8)
    r, p = scipy.stats.pearsonr(sub["daily_vol"], sub["daily_paid_usd"])
    print(f"{label_flat:<32}  r={r:+.3f}  p={p:.3g}  n={len(sub)}")

ax.set_xlabel("ETH Daily Volatility")
ax.set_ylabel("Daily Total Paid Bid (USD)")
ax.set_title("Arbitrum Revenue vs ETH Volatility â€” by Zone")
ax.legend(fontsize=9)
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("figures/paid_bid_vs_vol_by_zone.pdf", bbox_inches="tight")
plt.show()


In [ ]:
auctions["date"] = auctions["round_start_time"].dt.date
auctions["bid_gap"] = auctions["top_bid_usd"] - auctions["paid_bid_usd"]
daily_bid_gap = auctions.groupby("date")["bid_gap"].mean().reset_index()
daily_gap_vol = daily_bid_gap.merge(daily_vol, on="date", how="inner")
df = daily_gap_vol.dropna(subset=["bid_gap", "daily_vol"])

pearson_r, pearson_p = scipy.stats.pearsonr(df["bid_gap"], df["daily_vol"])
spearman_r, spearman_p = scipy.stats.spearmanr(df["bid_gap"], df["daily_vol"])

print("Correlation between Daily Bid Gap and Daily Volatility:")
print(f"Pearson:  r={pearson_r:.4f}, p={pearson_p:.4g}")
print(f"Spearman: r={spearman_r:.4f}, p={spearman_p:.4g}")

In [ ]:
# Daily mean bid gap vs ETH volatility, by zone
df_bg = daily_bid_gap.merge(daily_vol, on="date", how="inner").dropna()
df_bg["zone"] = df_bg["date"].apply(_assign_zone)

fig, ax = plt.subplots(figsize=(8, 5))

print("Daily Mean Bid Gap vs ETH Daily Volatility â€” by Zone")
print(f"{'Zone':<32}  {'r':>7}  {'p':>8}  n")
print("-" * 57)

for (z_start, z_end, z_label), z_color in zip(ZONE_LABELS, ZONE_COLORS):
    sub = df_bg[df_bg["zone"] == z_label]
    if len(sub) < 2:
        continue
    label_flat = z_label.replace("\n", " ")
    ax.scatter(sub["daily_vol"], sub["bid_gap"],
               label=label_flat, color=z_color, edgecolors="0.35",
               linewidths=0.5, s=60, alpha=0.9, zorder=3)
    m, b = np.polyfit(sub["daily_vol"], sub["bid_gap"], 1)
    xs = np.linspace(sub["daily_vol"].min(), sub["daily_vol"].max(), 100)
    ax.plot(xs, m * xs + b, color=z_color, linewidth=1.8)
    r, p = scipy.stats.pearsonr(sub["daily_vol"], sub["bid_gap"])
    print(f"{label_flat:<32}  r={r:+.3f}  p={p:.3g}  n={len(sub)}")

ax.set_xlabel("ETH Daily Volatility")
ax.set_ylabel("Daily Mean Bid Gap (USD)")
ax.set_title("Bid Gap vs ETH Volatility â€” by Zone")
ax.legend(fontsize=9)
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("figures/bid_gap_vs_vol_by_zone.pdf", bbox_inches="tight")
plt.show()


## PnL (pre-computed)
Load mark-to-market PnL from `searcher_txs_with_pnl_2.csv` â€” regenerate with `python compute_pnl.py`.

In [ ]:
sample_txs = pd.read_csv("searcher_txs_with_pnl_2.csv")
sample_txs["block_time"] = pd.to_datetime(sample_txs["block_time"], utc=True)

pnl_cols = [c for c in sample_txs.columns if c.startswith("pnl_t")]
pnl_data  = sample_txs[["tx_hash"] + pnl_cols].drop_duplicates("tx_hash")

def _merge_pnl(df):
    return (df.drop(columns=[c for c in pnl_cols if c in df.columns])
              .merge(pnl_data, on="tx_hash", how="left"))

wm_txs  = _merge_pnl(wm_txs)
sel_txs = _merge_pnl(sel_txs)

print(f"PnL columns loaded: {pnl_cols}")
print(f"wm_txs: {len(wm_txs):,} rows  |  sel_txs: {len(sel_txs):,} rows")


## Daily PnL Gaps
Analyze the relationship between ETH volatility and PnL advantages.

In [ ]:
def compute_daily_pnl_gaps(txs, daily_vol):
    """
    txs: filtered tx dataset for a single actor (wm_txs, sel_txs, ...)
    daily_vol: dataframe with "date" and "daily_vol"
    
    Returns: dict of {horizon -> merged df with tb, ntb, gap, vol}
    """
    if "date" not in txs:
        txs = txs.copy()
        txs["date"] = txs["block_time"].dt.date

    daily_summary = {}

    for h in HORIZONS:
        col = f"pnl_t{h}"
        df_pos = txs[txs[col] > 0]

        df = (
            df_pos.groupby(["date", "timeboosted"])[col]
            .sum()
            .reset_index()
            .rename(columns={col: f"pnl_t{h}"})
        )

        pivot = (
            df.pivot(index="date", columns="timeboosted", values=f"pnl_t{h}")
              .rename(columns={True: f"tb_pnl_t{h}", False: f"ntb_pnl_t{h}"})
              .reset_index()
        )

        merged = pivot.merge(daily_vol, on="date", how="left")
        merged[f"gap_t{h}"] = merged[f"tb_pnl_t{h}"] - merged[f"ntb_pnl_t{h}"]

        daily_summary[h] = merged

    return daily_summary

In [ ]:
wm_summary = compute_daily_pnl_gaps(wm_txs, daily_vol)
sel_summary = compute_daily_pnl_gaps(sel_txs, daily_vol)

In [ ]:
def correlations_to_latex(wm_summary, sel_summary):
    rows = []

    header = (
        "\\begin{table}[h]\n"
        "\\centering\n"
        "\\begin{tabular}{c|cc|cc}\n"
        "\\toprule\n"
        "Horizon & WM r & WM p & Selini r & Selini p \\\\\n"
        "\\midrule\n"
    )

    for h in HORIZONS:
        df_wm  = wm_summary[h].dropna(subset=["daily_vol", f"gap_t{h}"])
        df_sel = sel_summary[h].dropna(subset=["daily_vol", f"gap_t{h}"])

        if len(df_wm) < 2:
            wm_r, wm_p = float("nan"), float("nan")
        else:
            wm_r, wm_p = scipy.stats.pearsonr(df_wm["daily_vol"], df_wm[f"gap_t{h}"])

        if len(df_sel) < 2:
            sel_r, sel_p = float("nan"), float("nan")
        else:
            sel_r, sel_p = scipy.stats.pearsonr(df_sel["daily_vol"], df_sel[f"gap_t{h}"])

        row = (
            f"{h}s & "
            f"{wm_r:.3f} & {wm_p:.3f} & "
            f"{sel_r:.3f} & {sel_p:.3f} \\\\"
        )
        rows.append(row)

    footer = (
        "\\bottomrule\n"
        "\\end{tabular}\n"
        "\\caption{Correlation between daily volatility and TB-NonTB PnL gap for Wintermute and Selini.}\n"
        "\\label{tab:vol_pnl_corr}\n"
        "\\end{table}"
    )

    return header + "\n".join(rows) + "\n" + footer


latex_code = correlations_to_latex(wm_summary, sel_summary)

with open("vola_corr_table.tex", "w") as f:
    f.write(latex_code)

In [ ]:
def plot_daily_pnl_grid(daily_summary, actor_name):
    fig, axes = plt.subplots(2, 3, figsize=(18, 8))
    axes = axes.flatten()

    for i, h in enumerate(HORIZONS):
        ax = axes[i]
        df = daily_summary[h].copy()

        tb_col  = f"tb_pnl_t{h}"
        ntb_col = f"ntb_pnl_t{h}"

        ax.plot(df["date"], df[tb_col],  marker="o", color="blue",   label="Timeboosted")
        ax.plot(df["date"], df[ntb_col], marker="o", color="orange", label="Non-Timeboosted")

        ax.set_title(f"{h}s Horizon")
        ax.set_xlabel("Date")
        ax.set_ylabel("PNL (USD)")
        ax.grid(True)
        ax.xaxis.set_major_locator(plt.MaxNLocator(4))
        ax.legend()

    for j in range(len(HORIZONS), len(axes)):
        axes[j].axis("off")

    fig.suptitle(f"{actor_name}: Daily Timeboosted vs Non-Timeboosted PnL", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

plot_daily_pnl_grid(wm_summary, "Wintermute")
plot_daily_pnl_grid(sel_summary, "Selini")

In [ ]:
h = 5  # 5-second horizon

wm_5  = wm_summary[h].dropna(subset=["daily_vol", f"gap_t{h}"]).sort_values("date")
sel_5 = sel_summary[h].dropna(subset=["daily_vol", f"gap_t{h}"]).sort_values("date")

fig, ax1 = plt.subplots(figsize=(10, 5))

# Left y-axis: PnL gap per entity
ax1.plot(wm_5["date"], wm_5[f"gap_t{h}"], label="Wintermute gap", color="tab:blue")
ax1.plot(sel_5["date"], sel_5[f"gap_t{h}"], label="Selini gap",   color="tab:orange")
ax1.set_xlabel("Date")
ax1.set_ylabel(f"TB Advantage (PnL diff t={h}s)")
ax1.tick_params(axis="y", labelcolor="black")

# Right y-axis: volatility (same daily_vol for both, so just one line)
ax2 = ax1.twinx()
vol = daily_vol[daily_vol["date"].isin(wm_5["date"])].sort_values("date")
ax2.plot(vol["date"], vol["daily_vol"], label="ETH Daily Vol", color="tab:green", linestyle="--")
ax2.set_ylabel("ETH Daily Volatility")
ax2.tick_params(axis="y", labelcolor="tab:green")

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

fig.autofmt_xdate()
plt.title("Daily PnL Gap (5s) vs ETH Daily Volatility")
plt.tight_layout()
# plt.show()

# fig.savefig("daily_pnl_gap_vs_volatility.pdf")



## Daily PnL Over Time (t=5s Markout)
Timeboosted vs Non-Timeboosted PnL per searcher, using 5-second markout horizon.

In [ ]:
def _add_vol_axis(ax, df, date_col="date"):
    """Overlay daily_vol on a twin y-axis."""
    ax2 = ax.twinx()
    ax2.plot(df[date_col], df["daily_vol"],
             color="grey", linewidth=1.2, linestyle=":", alpha=0.7, label="ETH Daily Vol")
    ax2.set_ylabel("ETH Daily Volatility", color="grey", fontsize=9)
    ax2.tick_params(axis="y", labelcolor="grey")
    ax2.yaxis.set_label_position("right")
    return ax2


def _shade_zones(ax, date_min, date_max):
    for i, (z_start, z_end, z_label) in enumerate(ZONE_LABELS):
        x0 = date_min if z_start is None else z_start.tz_convert(None).to_datetime64()
        x1 = date_max if z_end   is None else z_end.tz_convert(None).to_datetime64()
        ax.axvspan(x0, x1, alpha=0.15, color=ZONE_COLORS[i], zorder=0)
        mid = pd.Timestamp(x0) + (pd.Timestamp(x1) - pd.Timestamp(x0)) / 2
        ylims = ax.get_ylim()
        ax.text(mid, ylims[1], z_label, ha="center", va="top",
                fontsize=7.5, color="0.3", style="italic")

h = 5

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=False)

searcher_summaries = [
    ("Wintermute", wm_summary, "#4C72B0"),
    ("Selini",     sel_summary, "#DD8452"),
]

for ax, (name, summary, base_color) in zip(axes, searcher_summaries):
    df = summary[h].copy()
    df["date"] = pd.to_datetime(df["date"])

    full_range = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
    df = df.set_index("date").reindex(full_range).reset_index().rename(columns={"index": "date"})

    ax.plot(df["date"], df[f"tb_pnl_t{h}"],  color=base_color, linewidth=1.8,
            marker="o", markersize=3, label="Timeboosted")
    ax.plot(df["date"], df[f"ntb_pnl_t{h}"], color=base_color, linewidth=1.8,
            marker="s", markersize=3, linestyle="--", alpha=0.6, label="Non-TB")
    ax.axhline(0, color="black", linewidth=0.6, linestyle=":")

    _shade_zones(ax, df["date"].min(), df["date"].max())
    ax2 = _add_vol_axis(ax, df)

    # combined legend
    lines1, labs1 = ax.get_legend_handles_labels()
    lines2, labs2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labs1 + labs2, fontsize=8)

    ax.set_title(name, fontsize=13)
    ax.set_xlabel("")
    ax.set_ylabel("Daily PnL (USD, positive trades only)")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

fig.suptitle(f"Daily Total PnL â€” {h}s Markout (TB vs Non-TB)", fontsize=14, y=1.02)
fig.tight_layout()
# plt.savefig("daily_pnl_over_time_t5.pdf", bbox_inches="tight", dpi=300)
plt.show()


## Average PnL Per Trade Per Day (t=5s Markout)
Mean per-trade PnL (positive trades only) by day, split by TB vs Non-TB and searcher.

In [ ]:
h = 5
pnl_col = f"pnl_t{h}"

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=False)

for ax, (name, addrs, base_color) in zip(axes, [
    ("Wintermute", SEARCHERS["Wintermute"], "#4C72B0"),
    ("Selini",     SEARCHERS["Selini"],     "#DD8452"),
]):
    df = txs[txs["tx_to_address"].isin(addrs)].copy()
    df["date"] = pd.to_datetime(df["block_time"]).dt.date

    avg = (
        df[df[pnl_col] > 0]
        .groupby(["date", "timeboosted"])[pnl_col]
        .mean()
        .unstack(fill_value=np.nan)
        .rename(columns={True: "tb_avg", False: "ntb_avg"})
        .reset_index()
    )
    avg["date"] = pd.to_datetime(avg["date"])

    full_range = pd.date_range(avg["date"].min(), avg["date"].max(), freq="D")
    avg = avg.set_index("date").reindex(full_range).reset_index().rename(columns={"index": "date"})

    # merge vol
    avg = avg.merge(daily_vol.assign(date=pd.to_datetime(daily_vol["date"])),
                    on="date", how="left")

    ax.plot(avg["date"], avg["tb_avg"],  color=base_color, linewidth=1.8,
            marker="o", markersize=3, label="Timeboosted avg")
    ax.plot(avg["date"], avg["ntb_avg"], color=base_color, linewidth=1.8,
            marker="s", markersize=3, linestyle="--", alpha=0.6, label="Non-TB avg")
    ax.axhline(0, color="black", linewidth=0.6, linestyle=":")

    _shade_zones(ax, avg["date"].min(), avg["date"].max())
    ax2 = _add_vol_axis(ax, avg)

    lines1, labs1 = ax.get_legend_handles_labels()
    lines2, labs2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labs1 + labs2, fontsize=8)

    ax.set_title(name, fontsize=13)
    ax.set_xlabel("")
    ax.set_ylabel("Avg PnL per trade (USD, positive trades only)")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

fig.suptitle(f"Daily Avg PnL per Trade â€” {h}s Markout (TB vs Non-TB)", fontsize=14, y=1.02)
fig.tight_layout()
plt.savefig("daily_avg_pnl_per_trade_t5.pdf", bbox_inches="tight", dpi=300)
plt.show()


## Total Surplus Over Time
Daily surplus = PnL(Wintermute + Selini, t=5s markout) âˆ’ total tx fees (all txs) âˆ’ bid paid (all auction winners).


In [ ]:
h = 5

only_pos = False

# ── 1. Daily searcher PnL (WM + Sel, all trades, t=5s) ──────────────────────
if only_pos:
    searcher_pnl = txs[txs[f"pnl_t{h}"] > 0].copy()
else:
    searcher_pnl = txs.copy()
searcher_pnl["date"] = pd.to_datetime(searcher_pnl["block_time"]).dt.date
daily_pnl = searcher_pnl.groupby("date")[f"pnl_t{h}"].sum().rename("total_pnl")

# ── 2. Daily tx fees (all txs) ───────────────────────────────────────────────
if only_pos:
    fees = txs[txs[f"pnl_t{h}"] > 0].copy()
else:
    fees = txs.copy()
fees["date"] = pd.to_datetime(fees["block_time"]).dt.date
daily_fees = fees.groupby("date")["tx_fee_usd"].sum().rename("total_fees")

# ── 3. Daily bid paid (all winners: WM, Sel, Kairos) ────────────────────────
auctions_s = auctions.copy()
auctions_s["date"] = auctions_s["round_start_time"].dt.tz_convert(None).dt.date
daily_bids = auctions_s.groupby("date")["paid_bid_usd"].sum().rename("total_bids_paid")

# ── 4. Merge and compute surplus ─────────────────────────────────────────────
surplus = (
    pd.DataFrame({"date": pd.date_range(
        start=min(daily_pnl.index.min(), daily_fees.index.min(), daily_bids.index.min()),
        end=max(daily_pnl.index.max(), daily_fees.index.max(), daily_bids.index.max()),
        freq="D",
    ).date})
    .set_index("date")
    .join(daily_pnl, how="left")
    .join(daily_fees, how="left")
    .join(daily_bids, how="left")
    .fillna(0)
    .reset_index()
)
surplus["date"] = pd.to_datetime(surplus["date"])
surplus["surplus"] = surplus["total_pnl"] - surplus["total_fees"] - surplus["total_bids_paid"]

total = surplus["total_pnl"].replace(0, float("nan"))
surplus["pct_surplus"] = surplus["surplus"]         / total * 100
surplus["pct_bids"]    = surplus["total_bids_paid"] / total * 100
surplus["pct_fees"]    = surplus["total_fees"]      / total * 100

vol_merged = surplus.merge(
    daily_vol.assign(date=pd.to_datetime(daily_vol["date"])), on="date", how="left"
)

# ── 5. Plot ──────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# --- top: zone shading only (no border lines, labels added after data is plotted) ---
date_min = surplus["date"].min()
date_max = surplus["date"].max()
for i, (z_start, z_end, _) in enumerate(ZONE_LABELS):
    x0 = date_min if z_start is None else z_start.tz_convert(None).to_datetime64()
    x1 = date_max if z_end   is None else z_end.tz_convert(None).to_datetime64()
    ax1.axvspan(x0, x1, alpha=0.15, color=ZONE_COLORS[i], zorder=0)

ax1.bar(surplus["date"], surplus["total_pnl"],        label="Searcher PnL",  color="#4C72B0", alpha=0.7, width=0.8)
ax1.bar(surplus["date"], -surplus["total_fees"],      label="−Tx Fees",      color="#DD8452", alpha=0.7, width=0.8)
ax1.bar(surplus["date"], -surplus["total_bids_paid"], label="−Bids Paid",    color="#C44E52", alpha=0.7, width=0.8,
        bottom=-surplus["total_fees"])
ax1.plot(surplus["date"], surplus["surplus"], color="black", linewidth=1.8,
         marker="o", markersize=3, label="Net Surplus", zorder=5)
ax1.axhline(0, color="black", linewidth=0.6, linestyle="-.")
ax1.set_ylabel("USD")
ax1.grid(axis="y", linestyle="--", alpha=0.4)

ax1v = ax1.twinx()
ax1v.plot(vol_merged["date"], vol_merged["daily_vol"],
          color="#2ca02c", linewidth=2, alpha=0.85, label="ETH Daily Vol")
ax1v.set_ylabel("ETH Daily Volatility", color="#2ca02c", fontsize=9)
ax1v.tick_params(axis="y", labelcolor="#2ca02c")

lines1, labs1 = ax1.get_legend_handles_labels()
lines2, labs2 = ax1v.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labs1 + labs2, fontsize=9)

# --- bottom: 100% stacked area ---
dates = surplus["date"]
y1 = surplus["pct_surplus"]
y2 = surplus["pct_surplus"] + surplus["pct_bids"]
y3 = surplus["pct_surplus"] + surplus["pct_bids"] + surplus["pct_fees"]

ax2.fill_between(dates, 0,  y1, label="Net Surplus", color="#4C72B0", alpha=0.8)
ax2.fill_between(dates, y1, y2, label="Bids Paid",   color="#C44E52", alpha=0.8)
ax2.fill_between(dates, y2, y3, label="Tx Fees",     color="#DD8452", alpha=0.8)
ax2.set_ylim(0, 100)
ax2.set_ylabel("% of PnL")
ax2.legend(fontsize=9, loc="upper left")
ax2.grid(axis="y", linestyle="--", alpha=0.4)

# zone labels for ax1 (after data plotted → correct ylim) + border lines + labels for ax2
ax1_ymax = ax1.get_ylim()[1]
for z_start, z_end, z_label in ZONE_LABELS:
    x0 = date_min if z_start is None else z_start.tz_convert(None).to_pydatetime()
    x1 = date_max if z_end   is None else z_end.tz_convert(None).to_pydatetime()
    mid = pd.Timestamp(x0) + (pd.Timestamp(x1) - pd.Timestamp(x0)) / 2
    ax1.text(mid, ax1_ymax, z_label.replace("\n", " "), ha="center", va="top",
             fontsize=7.5, color="0.25", style="italic", zorder=6)
    if z_start is not None:
        ax2.axvline(x0, color="0.25", linewidth=1.3, linestyle="--", zorder=5)
    ax2.text(mid, 97, z_label.replace("\n", " "), ha="center", va="top",
             fontsize=7.5, color="0.25", style="italic", zorder=6)

ax1.tick_params(axis="x", rotation=45)
ax2.tick_params(axis="x", rotation=45)

if only_pos:
    title_suffix = " (Positive PnL Trades Only)"
else:
    title_suffix = " (All Trades)"
fig.suptitle(f"Total Daily Surplus (PnL − Fees − Bids Paid), t={h}s Markout, {title_suffix}", fontsize=14)
fig.tight_layout()
plt.savefig(f"figures/total_surplus_over_time_{h}s_{title_suffix.strip(' ()')}.pdf", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
from scipy import stats

# vol_merged already has: surplus, total_pnl, daily_vol, date
df_corr = vol_merged.dropna(subset=["daily_vol", "surplus", "total_pnl"]).copy()
df_corr["zone"] = df_corr["date"].apply(_assign_zone)

def _corr_row(x, y, label_x, label_y, subset="All"):
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    r, p_r = stats.pearsonr(x, y)
    rho, p_rho = stats.spearmanr(x, y)
    return dict(subset=subset, x=label_x, y=label_y,
                n=len(x), pearson_r=r, p_pearson=p_r,
                spearman_rho=rho, p_spearman=p_rho)

rows = []
for y_col, y_label in [("surplus", "Net Surplus"), ("total_pnl", "Searcher PnL")]:
    rows.append(_corr_row(df_corr["daily_vol"], df_corr[y_col], "ETH Daily Vol", y_label))
    for zone, grp in df_corr.groupby("zone"):
        rows.append(_corr_row(grp["daily_vol"], grp[y_col], "ETH Daily Vol", y_label, subset=zone))

df_res = (pd.DataFrame(rows)[["y", "subset", "n", "pearson_r", "p_pearson", "spearman_rho", "p_spearman"]]
          .round({"pearson_r": 3, "p_pearson": 3, "spearman_rho": 3, "p_spearman": 3}))
print("Correlation: ETH Daily Volatility vs Surplus / PnL")
print(df_res.to_string(index=False))

# ── Scatter plots ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (y_col, y_label) in zip(axes, [("surplus", "Net Surplus (USD)"), ("total_pnl", "Searcher PnL (USD)")]):
    for (z_start, z_end, z_label), z_color in zip(ZONE_LABELS, ZONE_COLORS):
        sub = df_corr[df_corr["zone"] == z_label]
        ax.scatter(sub["daily_vol"], sub[y_col], label=z_label, color=z_color,
                   edgecolors="0.35", linewidths=0.5, alpha=0.8, s=40)
    # overall OLS line
    x_all = df_corr["daily_vol"].values
    y_all = df_corr[y_col].values
    m, b, r, p, _ = stats.linregress(x_all, y_all)
    x_line = np.linspace(x_all.min(), x_all.max(), 200)
    ax.plot(x_line, m * x_line + b, color="black", linewidth=1.5, linestyle="--",
            label=f"OLS (r={r:.2f}, p={p:.3f})")
    ax.axhline(0, color="0.5", linewidth=0.6, linestyle=":")
    ax.set_xlabel("ETH Daily Volatility")
    ax.set_ylabel(y_label)
    ax.set_title(y_label)
    ax.legend(fontsize=8)
    ax.grid(linestyle="--", alpha=0.4)

fig.suptitle(f"ETH Volatility vs Surplus / Searcher PnL (t={h}s markout)", fontsize=13)
fig.tight_layout()
plt.savefig(f"figures/surplus_vol_correlation_{h}s_{title_suffix.strip(' ()')}.pdf", bbox_inches="tight", dpi=300)
plt.show()

## PnL Summary & Comparison Tables
Aggregate PnL statistics for Wintermute and Selini, and export LaTeX tables.

In [ ]:
def compute_actor_summary(df, auctions):
    out = {}

    tb  = df[df["timeboosted"] == True]
    ntb = df[df["timeboosted"] == False]

    out["tb_count"]  = len(tb)
    out["ntb_count"] = len(ntb)
    out["tb_vol"]    = tb["amount_usd"].sum()
    out["ntb_vol"]   = ntb["amount_usd"].sum()

    out["tb_paid_usd"] = auctions[
        auctions["auction_round"].isin(tb["auction_round"].unique())
    ]["paid_bid_usd"].sum()

    for h in HORIZONS:
        col = f"pnl_t{h}"
        df_pos = df[df[col] > 0]

        out[f"tb_pnl_t{h}"]      = df_pos[df_pos["timeboosted"] == True][col].sum()
        out[f"ntb_pnl_t{h}"]     = df_pos[df_pos["timeboosted"] == False][col].sum()
        out[f"tb_avg_pnl_t{h}"]  = df_pos[df_pos["timeboosted"] == True][col].mean()
        out[f"ntb_avg_pnl_t{h}"] = df_pos[df_pos["timeboosted"] == False][col].mean()

    return out


wm_summary_stats  = compute_actor_summary(wm_txs, auctions)
sel_summary_stats = compute_actor_summary(sel_txs, auctions)

In [ ]:
def build_combined_table(wm_stats, sel_stats):
    rows = []

    rows.append(r"\textbf{Total TB Trades} & "
                f"{wm_stats['tb_count']:,} & {sel_stats['tb_count']:,} \\\\")
    rows.append(r"\textbf{Total Non-TB Trades} & "
                f"{wm_stats['ntb_count']:,} & {sel_stats['ntb_count']:,} \\\\")
    rows.append(r"\textbf{Total TB Volume (USD)} & "
                f"${wm_stats['tb_vol']:,.2f}$ & ${sel_stats['tb_vol']:,.2f}$ \\\\")
    rows.append(r"\textbf{Total Non-TB Volume (USD)} & "
                f"${wm_stats['ntb_vol']:,.2f}$ & ${sel_stats['ntb_vol']:,.2f}$ \\\\")
    rows.append(r"\textbf{Total Auction Bids Paid (USD)} & "
                f"${wm_stats['tb_paid_usd']:,.2f}$ & ${sel_stats['tb_paid_usd']:,.2f}$ \\\\")

    rows.append(r"\midrule")

    for h in HORIZONS:
        rows.append(
            rf"\textbf{{TB PnL @ {h}s}} & "
            f"${wm_stats[f'tb_pnl_t{h}']:,.2f}$ & "
            f"${sel_stats[f'tb_pnl_t{h}']:,.2f}$ \\\\"
        )
        rows.append(
            rf"\quad Avg PnL & "
            f"${wm_stats[f'tb_avg_pnl_t{h}']:,.2f}$ & "
            f"${sel_stats[f'tb_avg_pnl_t{h}']:,.2f}$ \\\\"
        )
        rows.append(
            rf"\textbf{{Non-TB PnL @ {h}s}} & "
            f"${wm_stats[f'ntb_pnl_t{h}']:,.2f}$ & "
            f"${sel_stats[f'ntb_pnl_t{h}']:,.2f}$ \\\\"
        )
        rows.append(
            rf"\quad Avg PnL & "
            f"${wm_stats[f'ntb_avg_pnl_t{h}']:,.2f}$ & "
            f"${sel_stats[f'ntb_avg_pnl_t{h}']:,.2f}$ \\\\"
        )
        rows.append(r"\addlinespace")

    latex = r"""
\begin{table}[ht]
\centering
\begin{tabular}{l rr}
\toprule
\textbf{Metric} & \textbf{Wintermute} & \textbf{Selini} \\
\midrule
""" + "\n".join(rows) + r"""
\bottomrule
\end{tabular}
\caption{Comparison of Wintermute and Selini Timeboosted vs Non-Timeboosted PnL, Averages, Volumes, and Trade Counts.}
\end{table}
"""
    return latex


combined_latex = build_combined_table(wm_summary_stats, sel_summary_stats)
with open(f"actor_comparison.tex", "w") as f:
    f.write(combined_latex)

In [ ]:
def build_pre_tb_table(wm_stats, sel_stats):
    rows = []

    rows.append(r"\textbf{Total Trades} & "
                f"{wm_stats['ntb_count']:,} & {sel_stats['ntb_count']:,} \\\\")
    rows.append(r"\textbf{Total Volume (USD)} & "
                f"${wm_stats['ntb_vol']:,.2f}$ & ${sel_stats['ntb_vol']:,.2f}$ \\\\")
    rows.append(r"\textbf{Total Auction Bids Paid (USD)} & "
                f"${wm_stats['tb_paid_usd']:,.2f}$ & ${sel_stats['tb_paid_usd']:,.2f}$ \\\\")

    rows.append(r"\midrule")

    for h in HORIZONS:
        rows.append(
            rf"\textbf{{PnL @ {h}s}} & "
            f"${wm_stats[f'ntb_pnl_t{h}']:,.2f}$ & "
            f"${sel_stats[f'ntb_pnl_t{h}']:,.2f}$ \\\\"
        )
        rows.append(
            rf"\quad Avg PnL & "
            f"${wm_stats[f'ntb_avg_pnl_t{h}']:,.2f}$ & "
            f"${sel_stats[f'ntb_avg_pnl_t{h}']:,.2f}$ \\\\"
        )
        rows.append(r"\addlinespace")

    latex = r"""
\begin{table}[ht]
\centering
\begin{tabular}{l rr}
\toprule
\textbf{Metric} & \textbf{Wintermute} & \textbf{Selini} \\
\midrule
""" + "\n".join(rows) + r"""
\bottomrule
\end{tabular}
\caption{PnL, averages, volumes, and trade counts for the pre-Timeboost period.}
\end{table}
"""
    return latex


pre_tb_latex = build_pre_tb_table(wm_summary_stats, sel_summary_stats)
with open("actor_comparison_pre_tb.tex", "w") as f:
    f.write(pre_tb_latex)

## Bid-PnL Correlation (Per Round)
Correlate auction bids (paid and top) with realized PnL at the round level across all horizons.

In [ ]:
def compute_entity_round_pnl(df, horizons, winner):
    """
    Compute PNL per auction round for a single entity,
    only counting timeboosted trades with positive PnL at each horizon.
    """
    results = []
    df = df[df["winner_name"] == winner]

    for round_id, g in df.groupby("auction_round"):
        row = {
            "auction_round": round_id,
            "winner_name": g["winner_name"].iloc[0],
            "top_bid_usd": g["top_bid_usd"].iloc[0],
            "paid_bid_usd": g["paid_bid_usd"].iloc[0],
            "round_start_time": g["round_start_time"].iloc[0],
            "round_end_time": g["round_end_time"].iloc[0],
        }

        for h in horizons:
            col = f"pnl_t{h}"
            # Filter independently per horizon (don't shrink g across iterations)
            g_pos = g[g[col] > 0]
            tb = g_pos[g_pos["timeboosted"] == True]
            row[f"tb_pnl_t{h}"] = tb[col].sum()

        results.append(row)

    return pd.DataFrame(results)

In [ ]:
wm_round_pnl = compute_entity_round_pnl(wm_txs, HORIZONS, "Wintermute")
sel_round_pnl = compute_entity_round_pnl(sel_txs, HORIZONS, "Selini")

In [ ]:
def compute_corr(df, bid_col, pnl_col="tb_pnl_t5"):
    df2 = df.dropna(subset=[bid_col, pnl_col])
    pear_r, pear_p = scipy.stats.pearsonr(df2[bid_col], df2[pnl_col])
    spear_r, spear_p = scipy.stats.spearmanr(df2[bid_col], df2[pnl_col])
    return pear_r, pear_p, spear_r, spear_p


corr_results = {
    "Wintermute": {
        "paid": compute_corr(wm_round_pnl, "paid_bid_usd"),
        "top":  compute_corr(wm_round_pnl, "top_bid_usd"),
    },
    "Selini": {
        "paid": compute_corr(sel_round_pnl, "paid_bid_usd"),
        "top":  compute_corr(sel_round_pnl, "top_bid_usd"),
    }
}

latex_table = r"""
\begin{table}[h!]
\centering
\renewcommand{\arraystretch}{1.3}
\begin{tabular}{lcccccccc}
\toprule
& \multicolumn{4}{c}{\textbf{Paid Bid vs PnL}} 
& \multicolumn{4}{c}{\textbf{Top Bid vs PnL}} \\
\cmidrule(lr){2-5} \cmidrule(lr){6-9}
\textbf{Entity} 
& \textbf{P-r} & \textbf{P-p} & \textbf{S-r} & \textbf{S-p}
& \textbf{P-r} & \textbf{P-p} & \textbf{S-r} & \textbf{S-p} \\
\midrule
Wintermute 
& %.4f & %.2g & %.4f & %.2g
& %.4f & %.2g & %.4f & %.2g \\
Selini 
& %.4f & %.2g & %.4f & %.2g
& %.4f & %.2g & %.4f & %.2g \\
\bottomrule
\end{tabular}
\caption{Correlation between auction bids (paid and top) and realized PnL for 5-second horizon.}
\end{table}
""" % (
    *corr_results["Wintermute"]["paid"],
    *corr_results["Wintermute"]["top"],
    *corr_results["Selini"]["paid"],
    *corr_results["Selini"]["top"]
)
with open("bid_pnl_corr.tex", "w") as f:
    f.write(latex_table)


In [ ]:
def corr_or_dash(df, bid_col, pnl_col, insignificant_marker="!"):
    """Return Pearson r or marker if non-significant."""
    df2 = df.dropna(subset=[bid_col, pnl_col])
    if len(df2) < 3:
        return insignificant_marker
    r, p = scipy.stats.pearsonr(df2[bid_col], df2[pnl_col])
    return f"{r:.3f}" if p < 0.05 else insignificant_marker

corr_table = {
    "Wintermute": {"paid": {}, "top": {}},
    "Selini":     {"paid": {}, "top": {}}
}

for h in HORIZONS:
    pnl_col = f"tb_pnl_t{h}"
    corr_table["Wintermute"]["paid"][h] = corr_or_dash(wm_round_pnl, "paid_bid_usd", pnl_col)
    corr_table["Wintermute"]["top"][h]  = corr_or_dash(wm_round_pnl, "top_bid_usd",  pnl_col)
    corr_table["Selini"]["paid"][h] = corr_or_dash(sel_round_pnl, "paid_bid_usd", pnl_col)
    corr_table["Selini"]["top"][h]  = corr_or_dash(sel_round_pnl, "top_bid_usd",  pnl_col)

latex = r"""
\begin{table}[h!]
\centering
\small
\renewcommand{\arraystretch}{1.25}
\begin{tabular}{l|cccccc|cccccc}
\toprule
& \multicolumn{6}{c|}{\textbf{Paid Bid vs PnL (Pearson r)}} 
& \multicolumn{6}{c}{\textbf{Top Bid vs PnL (Pearson r)}} \\
\cmidrule(lr){2-7} \cmidrule(lr){8-13}
\textbf{Entity} 
& 0s & 5s & 10s & 300s & 600s & 1800s
& 0s & 5s & 10s & 300s & 600s & 1800s \\
\midrule
"""

for entity in ["Wintermute", "Selini"]:
    row_paid = [corr_table[entity]["paid"][h] for h in HORIZONS]
    row_top  = [corr_table[entity]["top"][h]  for h in HORIZONS]

    latex += (
        entity + " & " +
        " & ".join(row_paid) + " & " +
        " & ".join(row_top) +
        r" \\" + "\n"
    )

latex += r"""\bottomrule
\end{tabular}
\caption{Correlation between bids (paid and top) and realized PnL across horizons. 
Values show Pearson $r$. ``!'' denotes non-significant results ($p > 0.05$).}
\end{table}
"""

with open("bid_pnl_correlation_all_horizons.tex", "w") as f:
    f.write(latex)

print("Saved LaTeX table to bid_pnl_correlation_all_horizons.tex")

In [ ]:
common_cols = [
    "auction_round", "paid_bid_usd", "top_bid_usd",
] + [f"tb_pnl_t{h}" for h in HORIZONS]

combined_round_pnl = pd.concat([
    wm_round_pnl[common_cols],
    sel_round_pnl[common_cols],
], ignore_index=True)

rows = []
for h in HORIZONS:
    pnl_col = f"tb_pnl_t{h}"
    rows.append({
        "horizon": h,
        "corr_paid": corr_or_dash(combined_round_pnl, "paid_bid_usd", pnl_col, "â€“"),
        "corr_top":  corr_or_dash(combined_round_pnl, "top_bid_usd",  pnl_col, "â€“"),
    })

combined_corr_df = pd.DataFrame(rows)
print(combined_corr_df)

## Submission Patterns

In [ ]:
def compute_tb_ratios(df):
    """Return dict[(winner_name, timeboosted)] -> ratio_within_tb."""
    counts = (
        df.groupby(["timeboosted", "winner_name"])
          .size()
          .reset_index(name="count")
    )
    totals = counts.groupby("timeboosted")["count"].transform("sum")
    counts["ratio"] = counts["count"] / totals

    ratios = {}
    for _, row in counts.iterrows():
        key = (row["winner_name"], bool(row["timeboosted"]))
        ratios[key] = row["ratio"]
    return ratios

wm_ratios  = compute_tb_ratios(wm_txs)
sel_ratios = compute_tb_ratios(sel_txs)

# Union of all winner names appearing in either df
winners = sorted({
    w for (w, tb) in wm_ratios.keys()
}.union({
    w for (w, tb) in sel_ratios.keys()
}))

lines = []
lines.append(r"\begin{table}[ht]")
lines.append(r"\centering")
lines.append(r"\begin{tabular}{lrrrr}")
lines.append(r"\toprule")
lines.append(r"Winner & WM TB & WM non-TB & Sel TB & Sel non-TB \\")
lines.append(r"\midrule")

for w in winners:
    wm_tb   = wm_ratios.get((w, True),  0.0) * 100
    wm_ntb  = wm_ratios.get((w, False), 0.0) * 100
    sel_tb  = sel_ratios.get((w, True), 0.0) * 100
    sel_ntb = sel_ratios.get((w, False),0.0) * 100

    line = "%s & %.1f\\%% & %.1f\\%% & %.1f\\%% & %.1f\\%% \\\\" % (
        w, wm_tb, wm_ntb, sel_tb, sel_ntb
    )
    lines.append(line)

lines.append(r"\bottomrule")
lines.append(r"\end{tabular}")
lines.append(r"\caption{Share of auction winners within timeboosted vs non-timeboosted trades for Wintermute and Selini.}")
lines.append(r"\label{tab:tb_winner_ratios}")
lines.append(r"\end{table}")

tex_content = "\n".join(lines)

with open("tb_winner_ratios.tex", "w") as f:
    f.write(tex_content)

## Time Since Round Start
Analyze transaction counts, PnL, and volume as a function of time elapsed since auction round start.

In [ ]:
pnl_col = "pnl_t5"   # adjust if needed

def group_by_time_and_tb(df, prefix, pnl_col="pnl_t5"):
    # # include only positive PnL trades
    # df = df[df[pnl_col] > 0]
    g = (
        df.groupby(["time_since_round_start", "timeboosted"])
          .agg(
              count=("tx_hash", "size"),
              pnl  =(pnl_col, "sum"),
              vol = ("amount_usd", "sum"),
              avg_pnl = (pnl_col, "mean"),
              avg_vol = ("amount_usd", "mean"),
          )
          .reset_index()
          .pivot(
              index="time_since_round_start",
              columns="timeboosted",
              values=["count", "pnl", "vol", "avg_pnl", "avg_vol"]
          )
    )

    # Flatten columns: ("count", True) â†’ "count_True"
    g.columns = [f"{a}_{b}" for a, b in g.columns]

    # Rename logically
    rename_map = {
        "count_True":  f"{prefix}_tb_count",
        "count_False": f"{prefix}_ntb_count",
        "pnl_True":    f"{prefix}_tb_pnl",
        "pnl_False":   f"{prefix}_ntb_pnl",
        "vol_True":    f"{prefix}_tb_vol",
        "vol_False":   f"{prefix}_ntb_vol",
        "avg_pnl_True": f"{prefix}_tb_avg_pnl",
        "avg_pnl_False": f"{prefix}_ntb_avg_pnl",
        "avg_vol_True": f"{prefix}_tb_avg_vol",
        "avg_vol_False": f"{prefix}_ntb_avg_vol",
    }
    g = g.rename(columns=rename_map)

    return g.reset_index()

wm_time_tb = group_by_time_and_tb(wm_txs, "wm", pnl_col=pnl_col)
sel_time_tb = group_by_time_and_tb(sel_txs, "sel", pnl_col=pnl_col)

In [ ]:
def _make_entity_cfg(prefix, df, color):
    cfg = {"df": df, "color": color}
    for metric in ["count", "pnl", "vol", "avg_pnl", "avg_vol"]:
        cfg[f"tb_{metric}"]  = f"{prefix}_tb_{metric}"
        cfg[f"ntb_{metric}"] = f"{prefix}_ntb_{metric}"
    return cfg

ENTITIES = {
    "WM":  _make_entity_cfg("wm",  wm_time_tb,  "blue"),
    "Sel": _make_entity_cfg("sel", sel_time_tb, "orange"),
}

In [ ]:
def plot_time_since_start_grid():
    metrics = [
        ("count", "Transaction Count",          "count_over_time"),
        ("pnl",   "PNL (USD)",                 "pnl_over_time"),
        ("vol",   "Trading Volume (USD)",      "vol_over_time"),
        ("avg_pnl", "Average PnL per Trade", "avg_pnl_over_time"),
        ("avg_vol", "Average Volume per Trade", "avg_vol_over_time"),
    ]

    fig, axes = plt.subplots(
        nrows=len(metrics), ncols=1, figsize=(8, 9), sharex=True, dpi=300
    )

    # One legend for the whole figure; collect handles as we go
    handles = []
    labels  = []

    for ax, (metric, ylabel, _) in zip(axes, metrics):
        for name, cfg in ENTITIES.items():
            df    = cfg["df"]
            color = cfg["color"]

            tb_col  = cfg[f"tb_{metric}"]
            ntb_col = cfg[f"ntb_{metric}"]

            # Solid = TB
            l1, = ax.plot(
                df["time_since_round_start"], df[tb_col],
                label=f"{name} TB", color=color, linestyle="-"
            )
            # Dashed = NTB
            l2, = ax.plot(
                df["time_since_round_start"], df[ntb_col],
                label=f"{name} non-TB", color=color, linestyle="--"
            )

            handles.extend([l1, l2])
            labels.extend([l1.get_label(), l2.get_label()])

        ax.set_ylabel(ylabel)
        ax.grid(True)

    axes[-1].set_xlabel("Time Since Auction Round Start (seconds)")

    # Global legend outside the plot area
    by_label = dict(zip(labels, handles))  # remove duplicates
    fig.legend(
        by_label.values(), by_label.keys(),
        loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.02)
    )

    fig.suptitle("Counts, PnL, and Volume Over Time Since Auction Round Start", y=1.06)
    fig.tight_layout()
    plt.show()
    # Optionally save:
    fig.savefig("time_since_round_start_grid.pdf", bbox_inches="tight")

# Run it
plot_time_since_start_grid()

In [ ]:
# Prepare non-searcher data for plotting
# Check if PnL columns exist for non_searcher_txs
has_pnl = "pnl_t5" in txs_non_searcher.columns

if has_pnl:
    # Use existing PnL column
    non_searcher_time_tb = group_by_time_and_tb(txs_non_searcher, "non_searcher", pnl_col="pnl_t5")
else:
    # Aggregate without PnL
    g = (
        txs_non_searcher.groupby(["time_since_round_start", "timeboosted"])
        .agg(
            count=("tx_hash", "size"),
            vol=("amount_usd", "sum"),
            avg_vol=("amount_usd", "mean"),
        )
        .reset_index()
        .pivot(
            index="time_since_round_start",
            columns="timeboosted",
            values=["count", "vol", "avg_vol"]
        )
    )
    
    # Flatten columns
    g.columns = [f"{a}_{b}" for a, b in g.columns]
    
    # Rename
    rename_map = {
        "count_True": "non_searcher_tb_count",
        "count_False": "non_searcher_ntb_count",
        "vol_True": "non_searcher_tb_vol",
        "vol_False": "non_searcher_ntb_vol",
        "avg_vol_True": "non_searcher_tb_avg_vol",
        "avg_vol_False": "non_searcher_ntb_avg_vol",
    }
    g = g.rename(columns=rename_map)
    non_searcher_time_tb = g.reset_index()

def plot_time_since_start_grid_non_searcher():
    """Plot transaction patterns for non-searcher transactions over time since auction round start."""
    
    # Check which metrics are available
    has_pnl = "non_searcher_tb_pnl" in non_searcher_time_tb.columns
    
    if has_pnl:
        metrics = [
            ("count", "Transaction Count"),
            ("pnl", "PNL (USD)"),
            ("vol", "Trading Volume (USD)"),
            ("avg_pnl", "Average PnL per Trade"),
            ("avg_vol", "Average Volume per Trade"),
        ]
    else:
        metrics = [
            ("count", "Transaction Count"),
            ("vol", "Trading Volume (USD)"),
            ("avg_vol", "Average Volume per Trade"),
        ]
    
    fig, axes = plt.subplots(
        nrows=len(metrics), ncols=1, figsize=(8, 3*len(metrics)), sharex=True, dpi=300
    )
    
    # Handle single subplot case
    if len(metrics) == 1:
        axes = [axes]
    
    # One legend for the whole figure
    handles = []
    labels = []
    
    for ax, (metric, ylabel) in zip(axes, metrics):
        tb_col = f"non_searcher_tb_{metric}"
        ntb_col = f"non_searcher_ntb_{metric}"
        
        # Check if columns exist
        if tb_col not in non_searcher_time_tb.columns:
            ax.text(0.5, 0.5, f"No data for {metric}", 
                   ha='center', va='center', transform=ax.transAxes)
            continue
        
        # Solid = TB
        l1, = ax.plot(
            non_searcher_time_tb["time_since_round_start"], 
            non_searcher_time_tb[tb_col],
            label="Non-Searcher TB", color="green", linestyle="-"
        )
        # Dashed = NTB
        l2, = ax.plot(
            non_searcher_time_tb["time_since_round_start"], 
            non_searcher_time_tb[ntb_col],
            label="Non-Searcher non-TB", color="green", linestyle="--"
        )
        
        handles.extend([l1, l2])
        labels.extend([l1.get_label(), l2.get_label()])
        
        ax.set_ylabel(ylabel)
        ax.grid(True)
    
    axes[-1].set_xlabel("Time Since Auction Round Start (seconds)")
    
    # Global legend
    by_label = dict(zip(labels, handles))
    fig.legend(
        by_label.values(), by_label.keys(),
        loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.02)
    )
    
    fig.suptitle("Non-Searcher Transactions: Counts and Volume Over Time Since Auction Round Start", y=1.04)
    fig.tight_layout()
    plt.show()
    # Optionally save:
    # fig.savefig("time_since_round_start_grid_non_searcher.pdf", bbox_inches="tight")

# Run it
plot_time_since_start_grid_non_searcher()

In [ ]:
def plot_time_since_start_comparison_wm_sel_non_searcher(only_positive_pnl=False, pnl_col="pnl_t5"):
    """
    Compare Wintermute, Selini, and Non-Searcher transactions over time since auction round start.
    Plots counts, volume, and Average Volume per Trade for TB and non-TB transactions.
    
    Parameters:
    -----------
    only_positive_pnl : bool
        If True, only include transactions with positive PnL
    pnl_col : str
        PnL column to use for filtering (default: "pnl_t5")
    """
    
    # Apply positive PnL filter if requested
    if only_positive_pnl:
        wm_filtered = wm_txs[wm_txs[pnl_col] > 0] if pnl_col in wm_txs.columns else wm_txs
        sel_filtered = sel_txs[sel_txs[pnl_col] > 0] if pnl_col in sel_txs.columns else sel_txs
        non_searcher_filtered = txs_non_searcher[txs_non_searcher[pnl_col] > 0] if pnl_col in txs_non_searcher.columns else txs_non_searcher
        
        # Recompute time-based aggregations with filtered data
        wm_time_filtered = group_by_time_and_tb(wm_filtered, "wm", pnl_col=pnl_col)
        sel_time_filtered = group_by_time_and_tb(sel_filtered, "sel", pnl_col=pnl_col)
        
        # For non-searchers, handle case where PnL might not exist
        if pnl_col in txs_non_searcher.columns:
            non_searcher_time_filtered = group_by_time_and_tb(non_searcher_filtered, "non_searcher", pnl_col=pnl_col)
        else:
            # If no PnL, use the original aggregation (no filtering possible)
            non_searcher_time_filtered = non_searcher_time_tb
        
        title_suffix = " (Positive PnL Only)"
        filename_suffix = "_positive_pnl"
    else:
        wm_time_filtered = wm_time_tb
        sel_time_filtered = sel_time_tb
        non_searcher_time_filtered = non_searcher_time_tb
        title_suffix = ""
        filename_suffix = ""
    
    # Check which metrics are available for others
    has_pnl = "non_searcher_tb_pnl" in non_searcher_time_filtered.columns
    
    if has_pnl:
        metrics = [
            ("count", "Transaction Count"),
            ("pnl", "PNL (USD)"),
            ("vol", "Trading Volume (USD)"),
            ("avg_pnl", "Average PnL per Trade"),
            ("avg_vol", "Average Volume per Trade"),
        ]
    else:
        # Only metrics available without PnL
        metrics = [
            ("count", "Transaction Count"),
            ("vol", "Trading Volume (USD)"),
            ("avg_vol", "Average Volume per Trade"),
        ]
    
    fig, axes = plt.subplots(
        nrows=len(metrics), ncols=1, figsize=(10, 3*len(metrics)), sharex=True, dpi=300
    )
    
    # Handle single subplot case
    if len(metrics) == 1:
        axes = [axes]
    
    # Configuration for each entity
    entities = [
        ("WM", wm_time_filtered, "wm", "blue"),
        ("Sel", sel_time_filtered, "sel", "orange"),
        ("Others", non_searcher_time_filtered, "non_searcher", "green"),
    ]
    
    handles = []
    labels = []
    
    for ax, (metric, ylabel) in zip(axes, metrics):
        for name, df, prefix, color in entities:
            tb_col = f"{prefix}_tb_{metric}"
            ntb_col = f"{prefix}_ntb_{metric}"
            
            # Check if columns exist (Others may not have all metrics)
            if tb_col not in df.columns or ntb_col not in df.columns:
                continue
            
            # Solid line = TB
            l1, = ax.plot(
                df["time_since_round_start"], 
                df[tb_col],
                label=f"{name} TB", 
                color=color, 
                linestyle="-",
                alpha=0.8
            )
            
            # Dashed line = non-TB
            l2, = ax.plot(
                df["time_since_round_start"], 
                df[ntb_col],
                label=f"{name} non-TB", 
                color=color, 
                linestyle="--",
                alpha=0.8
            )
            
            handles.extend([l1, l2])
            labels.extend([l1.get_label(), l2.get_label()])
        
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.3)
    
    axes[-1].set_xlabel("Time Since Auction Round Start (seconds)")
    
    # Global legend with unique labels
    by_label = dict(zip(labels, handles))
    fig.legend(
        by_label.values(), by_label.keys(),
        loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.02)
    )
    
    fig.suptitle(
        f"Comparison: WM, Selini & Others Over Time Since Auction Round Start{title_suffix}",
        y=1.05, fontsize=13
    )
    fig.tight_layout()
    plt.show()
    
    # Optionally save
    fig.savefig(f"time_since_round_start_comparison_all{filename_suffix}.pdf", bbox_inches="tight")

# Run the comparison plot with all trades
plot_time_since_start_comparison_wm_sel_non_searcher(only_positive_pnl=False)

# Also generate version with only positive PnL trades
plot_time_since_start_comparison_wm_sel_non_searcher(only_positive_pnl=True)

In [ ]:
fig, axes = plt.subplots(1, len(SEARCHERS), figsize=(7 * len(SEARCHERS), 4), sharey=False)
if len(SEARCHERS) == 1:
    axes = [axes]

for ax, (name, addrs) in zip(axes, SEARCHERS.items()):
    df = txs[txs["tx_to_address"].isin(addrs)].copy()
    df["date"] = df["block_time"].dt.date

    daily = df.groupby(["date", "timeboosted"]).size().unstack(fill_value=0)
    daily.columns = ["Non-TB" if not c else "Timeboosted" for c in daily.columns]
    daily.plot(kind="bar", ax=ax, color={"Non-TB": "steelblue", "Timeboosted": "tomato"},
               width=0.8, alpha=0.85, zorder=3)

    # Map dates to bar x-positions
    dates = list(daily.index)
    n = len(dates)

    def date_to_x(ts):
        """Return the x position (float) for a given tz-aware timestamp boundary."""
        d = ts.tz_convert(None).date()
        # Find the first bar index that is >= this date
        for i, bar_date in enumerate(dates):
            if bar_date >= d:
                return i - 0.5
        return n - 0.5

    # Shade regions and add labels
    for i, (z_start, z_end, z_label) in enumerate(ZONE_LABELS):
        x0 = -0.5          if z_start is None else date_to_x(z_start)
        x1 = n - 0.5       if z_end   is None else date_to_x(z_end)
        ax.axvspan(x0, x1, alpha=0.18, color=ZONE_COLORS[i], zorder=0)
        ax.text((x0 + x1) / 2, ax.get_ylim()[1],
                z_label, ha="center", va="bottom", fontsize=7.5,
                color="0.3", style="italic")

    ax.set_title(name, fontsize=13)
    ax.set_xlabel("")
    ax.set_ylabel("Tx count")
    ax.tick_params(axis="x", rotation=45)
    ax.legend()

fig.suptitle("Daily TB vs Non-TB Tx Count per Searcher", fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

## Reverts & Success Rates
Load revert data, compute success/failure rates, and compare CEX-DEX shares across timeboosted vs non-timeboosted transactions.

In [ ]:
csv_files = glob.glob(f"data/case_study/reverts/*_parsed.csv")
reverts = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

reverts["block_time"] = pd.to_datetime(reverts["block_time"], utc=True)
auctions["round_start_time"] = pd.to_datetime(auctions["round_start_time"], utc=True)
auctions["round_end_time"]   = pd.to_datetime(auctions["round_end_time"], utc=True)

# Remove timezone for merge_asof compatibility
reverts["block_time"]        = reverts["block_time"].dt.tz_convert(None)
auctions["round_start_time"] = auctions["round_start_time"].dt.tz_convert(None)
auctions["round_end_time"]   = auctions["round_end_time"].dt.tz_convert(None)

reverts = reverts.sort_values("block_time")
auctions = auctions.sort_values("round_start_time")

reverts["minute_start"] = reverts["block_time"].dt.floor("min")
reverts["time_since_minute_start"] = (
    reverts["block_time"] - reverts["minute_start"]
).dt.total_seconds()

reverts["timeboosted"] = reverts["timeboosted"].fillna(False)

In [ ]:
all_wm_txs = reverts[reverts["tx_to_address"].isin(SEARCHERS["Wintermute"])].copy()
all_sel_txs = reverts[reverts["tx_to_address"].isin(SEARCHERS["Selini"])].copy()

wm_success_not_cex_dex = all_wm_txs[(all_wm_txs["success"] == 1) & (~all_wm_txs["tx_hash"].isin(wm_txs["tx_hash"]))]
sel_success_not_cex_dex = all_sel_txs[(all_sel_txs["success"] == 1) & (~all_sel_txs["tx_hash"].isin(sel_txs["tx_hash"]))]

wm_tb = all_wm_txs[all_wm_txs["timeboosted"] == True]
wm_non_tb = all_wm_txs[all_wm_txs["timeboosted"] == False]
sel_tb = all_sel_txs[all_sel_txs["timeboosted"] == True]
sel_non_tb = all_sel_txs[all_sel_txs["timeboosted"] == False]

# Boolean flags for filtering
for df, cexdex_txs in [(all_wm_txs, wm_txs), (all_sel_txs, sel_txs)]:
    df["is_tb"]          = df["timeboosted"] == True
    df["is_ntb"]         = df["timeboosted"] == False
    df["is_success"]     = df["success"] == 1
    df["is_revert"]      = ~df["is_success"]
    df["is_cexdex"]      = df["tx_hash"].isin(cexdex_txs["tx_hash"])
    df["is_non_cexdex"]  = df["is_success"] & (~df["is_cexdex"])

summary = {}
for name, all_txs, cexdex_txs, tb, ntb in [
    ("Wintermute", all_wm_txs, wm_txs, wm_tb, wm_non_tb),
    ("Selini",     all_sel_txs, sel_txs, sel_tb, sel_non_tb),
]:
    summary[name] = {
        "cexdex":     len(cexdex_txs),
        "non_cexdex": len(all_txs[(all_txs["success"] == 1) & (~all_txs["tx_hash"].isin(cexdex_txs["tx_hash"]))]),
        "tb_total":   len(tb),
        "tb_rate":    tb["success"].mean(),
        "ntb_total":  len(ntb),
        "ntb_rate":   ntb["success"].mean(),
        "tb_cexdex":  len(tb[tb["tx_hash"].isin(cexdex_txs["tx_hash"])]),
        "ntb_cexdex": len(ntb[ntb["tx_hash"].isin(cexdex_txs["tx_hash"])]),
    }

In [ ]:
print(reverts['tx_to_address'].value_counts().head(20))

In [ ]:
searcher_colors = {"Wintermute": "#4C72B0", "Selini": "#DD8452"}

daily_parts = []
for name, df in {"Wintermute": all_wm_txs, "Selini": all_sel_txs}.items():
    d = df.copy()
    d["date"] = pd.to_datetime(d["block_time"]).dt.normalize()
    g = d.groupby("date").agg(total=("success", "count"), success=("success", "sum")).reset_index()
    g["reverts"]  = g["total"] - g["success"]
    g["searcher"] = name
    daily_parts.append(g)

daily_sr = pd.concat(daily_parts, ignore_index=True)

date_min = daily_sr["date"].min()
date_max = daily_sr["date"].max()

fig, ax = plt.subplots(figsize=(14, 5))

for i, (z_start, z_end, _) in enumerate(ZONE_LABELS):
    x0 = date_min if z_start is None else z_start.tz_convert(None).to_datetime64()
    x1 = date_max if z_end   is None else z_end.tz_convert(None).to_datetime64()
    ax.axvspan(x0, x1, alpha=0.12, color=ZONE_COLORS[i], zorder=0)

for name, grp in daily_sr.groupby("searcher"):
    grp = grp.sort_values("date")
    c = searcher_colors[name]
    ax.plot(grp["date"], grp["success"], color=c, linewidth=1.6,
            marker="o", markersize=2.5, linestyle="-",  label=f"{name} Success")
    ax.plot(grp["date"], grp["reverts"], color=c, linewidth=1.6,
            marker="o", markersize=2.5, linestyle="--", label=f"{name} Reverts", alpha=0.7)

ymax = ax.get_ylim()[1]
for z_start, z_end, z_label in ZONE_LABELS:
    x0 = date_min if z_start is None else z_start.tz_convert(None).to_pydatetime()
    x1 = date_max if z_end   is None else z_end.tz_convert(None).to_pydatetime()
    mid = pd.Timestamp(x0) + (pd.Timestamp(x1) - pd.Timestamp(x0)) / 2
    ax.text(mid, ymax, z_label, ha="center", va="top",
            fontsize=7.5, color="0.25", style="italic", zorder=6)

ax.set_ylabel("Daily Tx Count")
ax.legend(fontsize=9)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.tick_params(axis="x", rotation=45)
fig.suptitle("Daily Success & Revert Counts by Searcher", fontsize=14)
fig.tight_layout()
plt.savefig("success_revert_over_time.pdf", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
def pct(x, total):
    """Return percentage string like '23.4\\%'."""
    if total == 0:
        return "0.0\\%"
    return f"{(x/total)*100:.1f}\\%"

def fmt(n):
    """Format with thousands separator."""
    return f"{n:,.0f}"

def save_tb_success_table(all_wm_txs, wm_txs,
                                 all_sel_txs, sel_txs,
                                 outpath="tb_success_table.tex"):

    def compute(all_tx, cexdex):
        total = len(all_tx)
        success = int(all_tx["success"].sum())
        fail = total - success

        # TB / NTB split
        tb = all_tx[all_tx["timeboosted"] == True]
        ntb = all_tx[all_tx["timeboosted"] == False]

        tb_total = len(tb)
        tb_success = int(tb["success"].sum())
        tb_fail = tb_total - tb_success

        ntb_total = len(ntb)
        ntb_success = int(ntb["success"].sum())
        ntb_fail = ntb_total - ntb_success

        # CEX-DEX trades total
        cexdex_count = len(cexdex)
        cexdex_rate = pct(cexdex_count, success)

        # TB vs NTB CEXâ€“DEX contributions
        tb_cexdex = len(tb[tb["tx_hash"].isin(cexdex["tx_hash"])])
        ntb_cexdex = len(ntb[ntb["tx_hash"].isin(cexdex["tx_hash"])])

        return {
            "total": fmt(total),
            "success": pct(success, total),
            "fail": pct(fail, total),

            "cexdex": fmt(cexdex_count),
            "cexdex_rate": cexdex_rate,

            "tb_total": fmt(tb_total),
            "tb_success": pct(tb_success, tb_total),
            "tb_fail": pct(tb_fail, tb_total),
            "tb_cexdex": fmt(tb_cexdex),
            "tb_cexdex_rate": pct(tb_cexdex, tb_success),

            "ntb_total": fmt(ntb_total),
            "ntb_success": pct(ntb_success, ntb_total),
            "ntb_fail": pct(ntb_fail, ntb_total),
            "ntb_cexdex": fmt(ntb_cexdex),
            "ntb_cexdex_rate": pct(ntb_cexdex, ntb_success),
        }

    wm = compute(all_wm_txs, wm_txs)
    sel = compute(all_sel_txs, sel_txs)

    latex = r"""
\begin{table}[ht]
\centering
\begin{tabular}{l rr}
\toprule
\textbf{Metric} & \textbf{Wintermute} & \textbf{Selini} \\
\midrule
\textbf{Total Transactions} &
%(wm_total)s & %(sel_total)s \\
\quad Success Rate &
%(wm_success)s & %(sel_success)s \\
\quad Fail Rate &
%(wm_fail)s & %(sel_fail)s \\
\textbf{CEX--DEX Trades} &
%(wm_cexdex)s & %(sel_cexdex)s \\
\quad Share of Successful &
%(wm_cexdex_rate)s & %(sel_cexdex_rate)s \\
\midrule
\textbf{Timeboosted Transactions} &
%(wm_tb_total)s & %(sel_tb_total)s \\
\quad Success Rate &
%(wm_tb_success)s & %(sel_tb_success)s \\
\quad Fail Rate &
%(wm_tb_fail)s & %(sel_tb_fail)s \\
\quad CEX--DEX Share (of TB Success) &
%(wm_tb_cexdex_rate)s & %(sel_tb_cexdex_rate)s \\
\addlinespace
\textbf{Non-Timeboosted Transactions} &
%(wm_ntb_total)s & %(sel_ntb_total)s \\
\quad Success Rate &
%(wm_ntb_success)s & %(sel_ntb_success)s \\
\quad Fail Rate &
%(wm_ntb_fail)s & %(sel_ntb_fail)s \\
\quad CEX--DEX Share (of NTB Success) &
%(wm_ntb_cexdex_rate)s & %(sel_ntb_cexdex_rate)s \\
\bottomrule
\end{tabular}
\caption{Success rates and CEX--DEX shares for Timeboosted and non-Timeboosted transactions of Wintermute and Selini.}
\label{tab:tb-success}
\end{table}
""" % {
        "wm_total": wm["total"], "sel_total": sel["total"],
        "wm_success": wm["success"], "sel_success": sel["success"],
        "wm_fail": wm["fail"], "sel_fail": sel["fail"],

        "wm_cexdex": wm["cexdex"], "sel_cexdex": sel["cexdex"],
        "wm_cexdex_rate": wm["cexdex_rate"], "sel_cexdex_rate": sel["cexdex_rate"],

        "wm_tb_total": wm["tb_total"], "sel_tb_total": sel["tb_total"],
        "wm_tb_success": wm["tb_success"], "sel_tb_success": sel["tb_success"],
        "wm_tb_fail": wm["tb_fail"], "sel_tb_fail": sel["tb_fail"],
        "wm_tb_cexdex_rate": wm["tb_cexdex_rate"], "sel_tb_cexdex_rate": sel["tb_cexdex_rate"],

        "wm_ntb_total": wm["ntb_total"], "sel_ntb_total": sel["ntb_total"],
        "wm_ntb_success": wm["ntb_success"], "sel_ntb_success": sel["ntb_success"],
        "wm_ntb_fail": wm["ntb_fail"], "sel_ntb_fail": sel["ntb_fail"],
        "wm_ntb_cexdex_rate": wm["ntb_cexdex_rate"], "sel_ntb_cexdex_rate": sel["ntb_cexdex_rate"],
    }

    with open(outpath, "w") as f:
        f.write(latex)

    print(f"âœ” LaTeX table saved to: {outpath}")
    
save_tb_success_table(
    all_wm_txs, wm_txs,
    all_sel_txs, sel_txs,
    outpath=f"tb_success_table.tex"
)
